# 📱 DjezzyBot — Colab Quick Run (T4 GPU)

Voice + text chatbot for Djezzy — Arabic / French / English / Darija.

1. `Exécution ▸ Modifier le type d'exécution ▸ **T4 GPU**`
2. Run **Step 1**, then **Step 2**. The bot launches — click the share link. ✅

Code is pulled from GitHub (`YacefMehdi/DjezzyBot`). **Everything below Step 2 is optional** (live refresh, tests, thesis metrics) and is not needed to use the bot.

> **One-time:** the repo is private, so add a read-only GitHub token as a Colab secret named **`GH_TOKEN`** (key icon in the left sidebar → *Add new secret* → enable **Notebook access**). Make the token at GitHub ▸ Settings ▸ Developer settings ▸ Fine-grained tokens (Contents: Read-only).

## Step 1 — Setup  (pull latest code + install)

Re-run to update after a `git push`. If the session was already running, do **Execution ▸ Restart session** first so Python loads the new code.

In [ ]:
# Step 1 — pull the latest code from the private GitHub repo, then install deps.
import os, sys, shutil, subprocess
from google.colab import userdata

OWNER, REPO_NAME = "YacefMehdi", "DjezzyBot"
PROJ = "/content/DjezzyBot"
CLEAN_URL = f"https://github.com/{OWNER}/{REPO_NAME}.git"

try:
    token = userdata.get("GH_TOKEN")
except Exception as e:
    raise SystemExit("No GH_TOKEN secret. Left sidebar ▸ key icon ▸ add GH_TOKEN "
                     "(read-only token) ▸ enable Notebook access ▸ re-run. "
                     f"({type(e).__name__})")
auth_url = f"https://{token}@github.com/{OWNER}/{REPO_NAME}.git"

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError((r.stderr or r.stdout).replace(token, "***"))
    return r.stdout

if os.path.isdir(os.path.join(PROJ, ".git")):
    print("Pulling latest commit…")
    _run(f"git -C {PROJ} reset --hard -q")          # GitHub wins over any in-Colab edits
    _run(f"git -C {PROJ} pull -q {auth_url} main")
else:
    if os.path.isdir(PROJ):
        shutil.rmtree(PROJ)
    print("Cloning the repo (first run)…")
    _run(f"git clone -q {auth_url} {PROJ}")
    _run(f"git -C {PROJ} remote set-url origin {CLEAN_URL}")  # don't keep the token on disk

os.chdir(PROJ); sys.path.insert(0, PROJ)
print("Code:", _run(f"git -C {PROJ} log -1 --oneline").strip())

print("Installing dependencies… (~2-3 min the first time)")
subprocess.run("pip install -q -r requirements.txt", shell=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — set Execution ▸ Change runtime type ▸ T4 GPU, then re-run")

## Step 2 — Launch the bot 🚀

Builds the search index from the cached data and serves the UI. Click the public **share** link it prints (text + voice).

In [ ]:
import app
app.main()

---
## Optional — not needed to run the bot

Run any of these only if you want them. The bot above already works from the cached data.

### Live refresh (re-crawl djezzy.dz)
Installs Chromium so the **Rafraîchir** button and the daily 03:00 job can re-scrape the site.

In [ ]:
# Optional — enable the LIVE scraper. ~150 MB Chromium download, one-time.
import subprocess
subprocess.run("playwright install chromium", shell=True)
subprocess.run("playwright install-deps chromium", shell=True)
import scraper
print(scraper.smoke_test())          # {'ok': True, ...} = scraper works
# Full re-crawl + reindex (up to ~45 min):
# import scheduler; print(scheduler.force_refresh())

### Tests & thesis metrics
Acceptance suite (14 checks), robustness suite (25 cross-lingual scenarios), and the quantitative metrics.

In [ ]:
# Optional — build the index handle `store`, then run the acceptance suite.
import scraper, indexer, test_scenarios
store = indexer.build_index(scraper.load_pages())
test_scenarios._summary(test_scenarios.run_all(store))
# Robustness (25 scenarios):
# import test_robustness; test_robustness._summary(test_robustness.run_all(store))

In [ ]:
# Optional — quantitative metrics for the thesis (~10-12 min on T4, prints progress).
# Needs `store` from the cell above.
import evaluate
metrics = evaluate.run_all(store, with_llm=True)

### Catalog extractor — automatic data-cleaning, validated on a bigger open Qwen

This is the **automatic offer-extraction** experiment. The daily scrape rebuilds the raw
pages, and an LLM re-extracts the clean structured tiers (`offers.json` / `roaming.json`)
instead of someone hand-cleaning them. Here we *validate the method* by running it through a
**bigger open Qwen** (Qwen-32B via Groq — open-source, same family as the 7B, so the
"open-source only" brief holds) and scoring its output against your hand-verified gold.

**Your curated data is safe — two guards:**
1. The default run writes **drafts** (`offers.generated.json` / `roaming.generated.json`); it does **not** touch `offers.json` / `roaming.json`.
2. Every run first snapshots the gold to `data/backups/<timestamp>/`, so even an accidental `--apply` is reversible.

> Restore gold anytime:
> ```python
> import glob, shutil, os
> latest = sorted(glob.glob("data/backups/*"))[-1]
> for f in ("offers.json", "roaming.json"):
>     shutil.copy2(os.path.join(latest, f), os.path.join("data", f))
> print("restored from", latest)
> ```

In [ ]:
# Optional (extractor — step A) — point the extractor at a bigger OPEN Qwen on Groq.
# Get a free key at https://console.groq.com  ▸ API Keys  (regenerate if the old one expired).
# The key is typed via getpass, so it is NOT stored in the notebook.
import os, getpass
os.environ["LLM_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["LLM_MODEL"]    = "qwen-2.5-32b"   # bigger open Qwen; change if Groq lists another id
if not os.environ.get("LLM_API_KEY"):
    os.environ["LLM_API_KEY"] = getpass.getpass("Groq API key (free, console.groq.com): ")

# Not sure which Qwen ids Groq currently serves? Uncomment to list the models your key can use:
# import json, urllib.request
# req = urllib.request.Request("https://api.groq.com/openai/v1/models",
#                              headers={"Authorization": f"Bearer {os.environ['LLM_API_KEY']}"})
# print([m["id"] for m in json.load(urllib.request.urlopen(req))["data"] if "qwen" in m["id"].lower()])

In [ ]:
# Optional (extractor — step B) — DRAFT run + score against your verified gold.
# build_catalog.py : backs up gold to data/backups/<ts>/, then writes *.generated.json
#                    DRAFTS. It does NOT overwrite offers.json/roaming.json (no --apply).
# score_catalog.py : diffs the draft vs gold — MATCH / MISSING / PHANTOM + tier-recall %.
!python build_catalog.py
!python score_catalog.py
# Only if the score proves the method (high recall, no phantom/missing) would you ever run:
#   !python build_catalog.py --apply        # overwrite the live catalogs (still backed up first)